In [ ]:
from collections import Counter
import subprocess
import os
from itertools import product
from glob import glob

# Sequence Preprocessing

##### Before running this code ensure you have the necessary dependencies installed. Install FastQC and Trimmomatic. You can either use:
##### conda install -c bioconda fastqc trimmomatic
##### brew install fastqc trimmomatic


In [ ]:


def run_fastqc(fastq_file, output_dir="fastqc_results", threads=4):
    """
    Run FastQC to assess quality of FASTQ files.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"🔍 Running FastQC on {fastq_file}...")
    subprocess.run([
        "/data/users3/razumah1/anaconda3/bin/fastqc",
        "-t", str(threads),
        "-o", output_dir,
        fastq_file
    ], check=True)
    
    print(f"✅ FastQC completed. Results saved in {output_dir}")


def run_trimmomatic(input_fastq, output_prefix="trimmed", threads=4,
                    adapter_file=None, sliding_window="4:20", minlen=50):
    """
    Run Trimmomatic for trimming sequencing reads.
    Supports single-end FASTQ files.
    """
    output_fastq = f"{output_prefix}.fastq.gz"
    log_file = f"{output_prefix}_trimmomatic.log"

    # Build Trimmomatic command
    cmd = [
        "trimmomatic", "SE",                      # single-end mode
        "-threads", str(threads),
        "-phred33",                               # adjust if your data is PHRED64
        input_fastq, output_fastq
    ]
    
    # Add trimming steps
    if adapter_file:
        cmd.append(f"ILLUMINACLIP:{adapter_file}:2:30:10")
    cmd.append(f"SLIDINGWINDOW:{sliding_window}")
    cmd.append(f"MINLEN:{minlen}")

    print(f"✂️ Running Trimmomatic on {input_fastq}...")
    with open(log_file, "w") as log:
        subprocess.run(cmd, stdout=log, stderr=log, check=True)

    print(f"✅ Trimmomatic completed. Clean FASTQ: {output_fastq}")
    return output_fastq


def preprocess_pipeline(fastq_file, adapter_file=None):
    """
    Full preprocessing pipeline: FastQC → Trimmomatic → FastQC again.
    """
    # Step 1: Quality check before trimming
    run_fastqc(fastq_file, output_dir="fastqc_before")

    # Step 2: Trimming
    cleaned_fastq = run_trimmomatic(
        fastq_file,
        output_prefix="cleaned_reads",
        adapter_file=adapter_file
    )

    # Step 3: Quality check after trimming
    run_fastqc(cleaned_fastq, output_dir="fastqc_after")

    return cleaned_fastq


In [ ]:


# Define directories
input_folder = "./data"               # Folder containing FASTQ files
output_folder = "./cleaned_data"      # Folder for cleaned outputs
os.makedirs(output_folder, exist_ok=True)

# Adapter file (Trimmomatic)
adapter_path = "/opt/homebrew/Cellar/trimmomatic/0.39_1/libexec/Trimmomatic-0.39/adapters/TruSeq3-SE.fa"

# Process all FASTQ files in the input folder
fastq_files = glob(os.path.join(input_folder, "*.fastq"))

for fastq_file in fastq_files:
    # Derive output file path
    base_name = os.path.basename(fastq_file)
    output_file = os.path.join(output_folder, f"cleaned_{base_name}")

    # Run your preprocessing pipeline
    cleaned_fastq = preprocess_pipeline(fastq_file, adapter_file=adapter_path)

    # Move or rename cleaned output to target folder (if preprocess_pipeline doesn't handle it)
    if cleaned_fastq and os.path.exists(cleaned_fastq):
        os.rename(cleaned_fastq, output_file)

    print(f"✅ Cleaned FASTQ saved to: {output_file}")

print("\n🎉 All FASTQ files have been processed and saved in 'cleaned_data' folder!")


🔍 Running FastQC on ./wastewater_seq_dataset.fastq...
null


Started analysis of wastewater_seq_dataset.fastq
Approx 5% complete for wastewater_seq_dataset.fastq
Approx 10% complete for wastewater_seq_dataset.fastq
Approx 15% complete for wastewater_seq_dataset.fastq
Approx 20% complete for wastewater_seq_dataset.fastq
Approx 25% complete for wastewater_seq_dataset.fastq
Approx 30% complete for wastewater_seq_dataset.fastq
Approx 35% complete for wastewater_seq_dataset.fastq
Approx 40% complete for wastewater_seq_dataset.fastq
Approx 45% complete for wastewater_seq_dataset.fastq
Approx 50% complete for wastewater_seq_dataset.fastq
Approx 55% complete for wastewater_seq_dataset.fastq
Approx 60% complete for wastewater_seq_dataset.fastq
Approx 65% complete for wastewater_seq_dataset.fastq
Approx 70% complete for wastewater_seq_dataset.fastq
Approx 75% complete for wastewater_seq_dataset.fastq
Approx 80% complete for wastewater_seq_dataset.fastq
Approx 85% complete for wastewater_seq_dataset.fastq
Approx 90% complete for wastewater_seq_dataset.fast

Analysis complete for wastewater_seq_dataset.fastq
✅ FastQC completed. Results saved in fastqc_before
✂️ Running Trimmomatic on ./wastewater_seq_dataset.fastq...
✅ Trimmomatic completed. Clean FASTQ: cleaned_reads.fastq.gz
🔍 Running FastQC on cleaned_reads.fastq.gz...
application/gzip


Started analysis of cleaned_reads.fastq.gz
Approx 5% complete for cleaned_reads.fastq.gz
Approx 10% complete for cleaned_reads.fastq.gz
Approx 15% complete for cleaned_reads.fastq.gz
Approx 20% complete for cleaned_reads.fastq.gz
Approx 25% complete for cleaned_reads.fastq.gz
Approx 30% complete for cleaned_reads.fastq.gz
Approx 35% complete for cleaned_reads.fastq.gz
Approx 40% complete for cleaned_reads.fastq.gz
Approx 45% complete for cleaned_reads.fastq.gz
Approx 50% complete for cleaned_reads.fastq.gz
Approx 55% complete for cleaned_reads.fastq.gz
Approx 60% complete for cleaned_reads.fastq.gz
Approx 65% complete for cleaned_reads.fastq.gz
Approx 70% complete for cleaned_reads.fastq.gz
Approx 75% complete for cleaned_reads.fastq.gz
Approx 80% complete for cleaned_reads.fastq.gz
Approx 85% complete for cleaned_reads.fastq.gz
Approx 90% complete for cleaned_reads.fastq.gz
Approx 95% complete for cleaned_reads.fastq.gz


Analysis complete for cleaned_reads.fastq.gz
✅ FastQC completed. Results saved in fastqc_after
Final cleaned FASTQ ready at: cleaned_reads.fastq.gz
